In [17]:
import pandas as pd
import requests
import json
import time
import datetime

In [2]:
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

,city
0,Mont Saint Michel
1,St Malo
2,Bayeux
3,Le Havre
4,Rouen


In [72]:
df = df_source.copy()
for index, row in df.iterrows():
    print(index, row["city"])
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
    res = requests.get(f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json", headers=headers)
    city = res.json()[0]
    df.loc[index, "lat"] = city["lat"]
    df.loc[index, "lon"] = city["lon"]
    time.sleep(1)
df.head()
df.to_csv("cities_with_geoposition.csv", index=False)

0 Mont Saint Michel
1 St Malo
2 Bayeux
3 Le Havre
4 Rouen
5 Paris
6 Amiens
7 Lille
8 Strasbourg
9 Chateau du Haut Koenigsbourg
10 Colmar
11 Eguisheim
12 Besancon
13 Dijon
14 Annecy
15 Grenoble
16 Lyon
17 Gorges du Verdon
18 Bormes les Mimosas
19 Cassis
20 Marseille
21 Aix en Provence
22 Avignon
23 Uzes
24 Nimes
25 Aigues Mortes
26 Saintes Maries de la mer
27 Collioure
28 Carcassonne
29 Ariege
30 Toulouse
31 Montauban
32 Biarritz
33 Bayonne
34 La Rochelle


In [3]:
df = pd.read_csv("cities_with_geoposition.csv")
df.head()

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966


In [38]:
list_weather_data = []

for index, row in df.iterrows():
    res_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid=4656ed7337e689999007412af6a4dafe", headers=headers)
    res_weather_json = res_weather.json()
    
    for res in res_weather_json['list']:
        weather_entry = {
            "city": row['city'],
            "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%d/%m/%y %H:%M'),
            "temp": res['main']['temp'],
            "perc_humidity": res['main']['humidity'],
            "prob_rain": res['pop'],
            "wind_speed": res['wind']['speed'],
            "perc_cloud": res['clouds']['all']
        }
        list_weather_data.append(weather_entry)
    # Stock score for temperature, rain prob, cloud prob, win speed
    # Scale all to the same scale
    # Calculate Score = (Température * Coeff) - (Pluie * Coeff) - (Nuages * Coeff)
        # Define the coef based on my own importance
    time.sleep(1)
    
df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())
df_weather.to_csv("weather_forecast.csv", index=False)

                city            date  temp  perc_humidity  prob_rain  \
0  Mont Saint Michel  09/02/26 16:00  8.79             93       1.00   
1  Mont Saint Michel  09/02/26 19:00  9.40             94       1.00   
2  Mont Saint Michel  09/02/26 22:00  9.44             95       0.41   
3  Mont Saint Michel  10/02/26 01:00  9.56             95       1.00   
4  Mont Saint Michel  10/02/26 04:00  9.71             91       0.55   

   wind_speed  perc_cloud  
0        9.97         100  
1        6.10         100  
2        4.61          91  
3        5.57          93  
4        6.50         100  
